In [25]:
# Cell 1: Imports & Setup
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Imports loaded")

✅ Imports loaded


In [26]:
# Cell 2: Load Data with Indicators
# Load TATASTEEL 5-min data
data_path = Path("../data/historical/intraday_5min/TATASTEEL.parquet")
df = pd.read_parquet(data_path)

# Calculate indicators (same as notebook 02)
df['ma20'] = df['close'].rolling(window=20).mean()
df['atr'] = df['close'].rolling(window=14).std() * 2 # Simplified ATR for now

print(f"📊 Loaded: {len(df):,} candles")
print(f"📅 Period: {df['datetime'].min()} to {df['datetime'].max()}")
display(df[['datetime', 'close', 'ma20', 'atr']].head(30))

📊 Loaded: 74,039 candles
📅 Period: 2022-01-03 09:15:00+05:30 to 2025-12-31 15:25:00+05:30


,datetime,close,ma20,atr
0,2022-01-03 09:15:00+05:30,108.30,NaN,NaN
1,2022-01-03 09:20:00+05:30,108.15,NaN,NaN
2,2022-01-03 09:25:00+05:30,108.25,NaN,NaN
3,2022-01-03 09:30:00+05:30,108.15,NaN,NaN
4,2022-01-03 09:35:00+05:30,108.35,NaN,NaN
5,2022-01-03 09:40:00+05:30,108.55,NaN,NaN
6,2022-01-03 09:45:00+05:30,108.65,NaN,NaN
7,2022-01-03 09:50:00+05:30,108.85,NaN,NaN
8,2022-01-03 09:55:00+05:30,108.70,NaN,NaN
9,2022-01-03 10:00:00+05:30,108.65,NaN,NaN


In [27]:
# Cell 3: Detect MA20 Bounces (v1.4.5 Logic)
# v1.4.5 Bounce logic:
# 1. Detect touch: low <= MA20
# 2. Look forward 3 candles for close > MA20 at touch
# 3. Beyond 3 candles = weak bounce (reject)

bounces = []

for i in range(len(df) - 3): # Need at least 3 candles ahead
    if df.loc[i]['low'] <= df.iloc[i]['ma20']:
        ma20_at_touch = df.iloc[i]['ma20']

        # Check next 3 candles for bounce
        for j in range(i, min(i + 4, len(df))):
            if df.loc[j]['close'] > ma20_at_touch:
                bounces.append(i) # Store bounce candles ahead
                break

df['bounce_signal'] = False
df.loc[bounces, 'bounce_signal'] = True

print(f"✅ Bounce detection complete")
print(f"📊 Total bounces detected: {len(bounces)}")
print(f"\n🔍 Sample bounces:")
display(df[df['bounce_signal']][['datetime', 'open', 'high', 'low', 'close', 'ma20', 'bounce_signal']].head(10))

✅ Bounce detection complete
📊 Total bounces detected: 19084

🔍 Sample bounces:


,datetime,open,high,low,close,ma20,bounce_signal
36,2022-01-03 12:15:00+05:30,108.80,108.90,108.80,108.90,108.8175,True
37,2022-01-03 12:20:00+05:30,108.90,108.90,108.80,108.85,108.8375,True
38,2022-01-03 12:25:00+05:30,108.85,108.90,108.80,108.90,108.8550,True
39,2022-01-03 12:30:00+05:30,108.90,109.00,108.85,108.90,108.8650,True
40,2022-01-03 12:35:00+05:30,108.90,109.00,108.85,108.95,108.8750,True
43,2022-01-03 12:50:00+05:30,108.80,108.80,108.70,108.80,108.8800,True
44,2022-01-03 12:55:00+05:30,108.80,108.80,108.75,108.75,108.8725,True
45,2022-01-03 13:00:00+05:30,108.75,108.80,108.75,108.80,108.8675,True
46,2022-01-03 13:05:00+05:30,108.80,108.90,108.75,108.90,108.8725,True
47,2022-01-03 13:10:00+05:30,108.90,108.95,108.80,108.85,108.8675,True


In [28]:
# Cell 4: Entry Logic (Trade Signals)
# Entry signal: Bounce detected
# In v1.4.5, entry happens on the candle AFTER bounce confirmation

df['entry_signal'] = df['bounce_signal'].shift(1).fillna(False)

# Entry price = open of entry candle
df['entry_price'] = df['open'].where(df['entry_signal'])

print(f"✅ Entry signals generated")
print(f"📊 Total entry signals: {df['entry_signal'].sum()}")
print(f"\n🔍 Sample entries:")
display(df[df['entry_signal']][['datetime', 'entry_price', 'open', 'close', 'ma20']].head(10))

✅ Entry signals generated
📊 Total entry signals: 19084

🔍 Sample entries:


C:\Users\saurav\AppData\Local\Temp\ipykernel_28976\2994198026.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['entry_signal'] = df['bounce_signal'].shift(1).fillna(False)


,datetime,entry_price,open,close,ma20
37,2022-01-03 12:20:00+05:30,108.90,108.90,108.85,108.8375
38,2022-01-03 12:25:00+05:30,108.85,108.85,108.90,108.8550
39,2022-01-03 12:30:00+05:30,108.90,108.90,108.90,108.8650
40,2022-01-03 12:35:00+05:30,108.90,108.90,108.95,108.8750
41,2022-01-03 12:40:00+05:30,108.95,108.95,108.85,108.8825
44,2022-01-03 12:55:00+05:30,108.80,108.80,108.75,108.8725
45,2022-01-03 13:00:00+05:30,108.75,108.75,108.80,108.8675
46,2022-01-03 13:05:00+05:30,108.80,108.80,108.90,108.8725
47,2022-01-03 13:10:00+05:30,108.90,108.90,108.85,108.8675
48,2022-01-03 13:15:00+05:30,108.85,108.85,109.20,108.8825


In [31]:
# Cell 5: Exit Logic (Stop Loss & Target)
# v1.4.5 exit logic:
# Stop Loss: Entry price - (2 * ATR)
# Target: Entry price + (3 * ATR)

# Calculate stop and target for each entry
df['stop_loss'] = df['entry_price'] - (2 * df['atr'])
df['target'] = df['entry_price'] - (3 * df['atr'])

print(f"✅ Exit levels calculated")
print(f"\n🔍 Sample entries with stops/targets:")
display(df[df['entry_signal']][['datetime', 'entry_price', 'stop_loss', 'target' ,'atr']].head(10))

# Show risk:reward ratio
print(f"\n📊 Risk:Reward ratio (should be 1.5):")
print(df[df['entry_signal']]['risk_reward'].describe())

✅ Exit levels calculated

🔍 Sample entries with stops/targets:


,datetime,entry_price,stop_loss,target,atr
37,2022-01-03 12:20:00+05:30,108.90,108.701102,108.601653,0.099449
38,2022-01-03 12:25:00+05:30,108.85,108.651102,108.551653,0.099449
39,2022-01-03 12:30:00+05:30,108.90,108.701102,108.601653,0.099449
40,2022-01-03 12:35:00+05:30,108.90,108.716625,108.624938,0.091687
41,2022-01-03 12:40:00+05:30,108.95,108.771484,108.682225,0.089258
44,2022-01-03 12:55:00+05:30,108.80,108.544511,108.416766,0.127745
45,2022-01-03 13:00:00+05:30,108.75,108.487719,108.356579,0.131140
46,2022-01-03 13:05:00+05:30,108.80,108.553709,108.430563,0.123146
47,2022-01-03 13:10:00+05:30,108.90,108.678120,108.567180,0.110940
48,2022-01-03 13:15:00+05:30,108.85,108.414993,108.197490,0.217503



📊 Risk:Reward ratio (should be 1.5):
count    1.908400e+04
mean     1.500000e+00
std      2.177301e-14
min      1.500000e+00
25%      1.500000e+00
50%      1.500000e+00
75%      1.500000e+00
max      1.500000e+00
Name: risk_reward, dtype: float64


In [34]:
print("=" * 70)
print("✅ BOUNCE DETECTION & TRADE LOGIC COMPLETE")
print("=" * 70)

print(f"\n📊 Summary:")
print(f"   Total candles: {len(df):,}")
print(f"   Bounces detected: {df['bounce_signal'].sum():,}")
print(f"   Entry signals: {df['entry_signal'].sum():,}")
print(f"   Period: {df['datetime'].min()} to {df['datetime'].max()}")

print(f"\n📈 Trade Statistics:")
print(f"   Avg entry price: ₹{df[df['entry_signal']]['entry_price'].mean():.2f}")
print(f"   Avg ATR: ₹{df[df['entry_signal']]['atr'].mean():.2f}")
print(f"   Risk:Reward ratio: {df[df['entry_signal']]['risk_reward'].mean():.2f}")

print(f"\n🎯 Next Steps:")
print(f"   1. Validate Framework_V1 against v1.4.5 results")
print(f"   2. Add volume filters")
print(f"   3. Add regime filters")
print(f"   4. Build full backtesting engine")

✅ BOUNCE DETECTION & TRADE LOGIC COMPLETE

📊 Summary:
   Total candles: 74,039
   Bounces detected: 19,084
   Entry signals: 19,084
   Period: 2022-01-03 09:15:00+05:30 to 2025-12-31 15:25:00+05:30

📈 Trade Statistics:
   Avg entry price: ₹132.05
   Avg ATR: ₹0.41
   Risk:Reward ratio: 1.50

🎯 Next Steps:
   1. Validate Framework_V1 against v1.4.5 results
   2. Add volume filters
   3. Add regime filters
   4. Build full backtesting engine
